# Phase 1 Task B — Results Driver

Sau khi cả 2 machine chạy xong 15 runs (5 models × 3 seeds), chạy notebook này
để tổng hợp kết quả từ wandb và tạo bảng tổng kết.

**Có thể chạy trên local / Colab / bất kỳ máy nào có internet.**

**Workflow:**
1. Pull wandb runs hoặc đọc checkpoints local
2. Evaluate mỗi model × seed trên test set
3. Tổng hợp F1, accuracy, confusion matrix
4. So sánh với Task A baselines

## 1. Setup & wandb Fetch

In [ ]:
# Install deps
import subprocess, sys
for pkg in ["wandb", "scikit-learn", "seaborn", "matplotlib", "timm", "albumentations"]:
    try: __import__(pkg.replace("-", "_"))
    except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import os, getpass
if "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_API_KEY"] = getpass.getpass("wandb API key: ")

import wandb
api = wandb.Api()
print("[wandb] API ready")

## 2. Fetch Runs from wandb

In [ ]:
PROJECT = "woodbench-taskB"
MODELS = ["convnext_tiny", "mobilenetv3_large", "swin_tiny", "resnet50", "efficientnet_b3"]
SEEDS = [42, 123, 456]

runs_data = {}

for model in MODELS:
    runs_data[model] = {}
    for seed in SEEDS:
        try:
            # Filter runs by name pattern
            runs = api.runs(
                PROJECT,
                filters={
                    "$and": [
                        {"config.backbone": model},
                        {"config.seed": seed}
                    ]
                }
            )
            if runs:
                run = runs[0]
                summary = run.summary._json
                runs_data[model][seed] = {
                    "name": run.name,
                    "best_val_acc": summary.get("val/acc", 0),
                    "final_val_acc": summary.get("val/acc"),
                    "run_id": run.id,
                    "state": run.state,
                }
                print(f"[OK] {model} s{seed}: {runs_data[model][seed]['best_val_acc']:.2f}%")
            else:
                print(f"[MISSING] {model} s{seed}")
        except Exception as e:
            print(f"[ERR] {model} s{seed}: {e}")

## 3. Summary Table

In [ ]:
import pandas as pd
import numpy as np

rows = []
for model in MODELS:
    for seed in SEEDS:
        d = runs_data[model].get(seed, {})
        rows.append({
            "Model": model,
            "Seed": seed,
            "Best Val Acc (%)": d.get("best_val_acc", 0),
            "State": d.get("state", "missing"),
        })

df = pd.DataFrame(rows)

# Group summary
summary = df.groupby("Model").agg(
    Mean_Acc=("Best Val Acc (%)", "mean"),
    Std_Acc=("Best Val Acc (%)", "std"),
    Min_Acc=("Best Val Acc (%)", "min"),
    Max_Acc=("Best Val Acc (%)", "max"),
).round(2)

print("\n" + "="*70)
print("TASK B — 5 MODELS × 3 SEEDS SUMMARY")
print("="*70)
print(df.to_string(index=False))
print("\n" + "-"*70)
print("PER-MODEL SUMMARY")
print("-"*70)
print(summary.to_string())
print("\n" + "="*70)

# Highlight best model
best_model = summary["Mean_Acc"].idxmax()
best_acc = summary.loc[best_model, "Mean_Acc"]
print(f"Best model: {best_model} ({best_acc:.2f}%)")

## 4. Export to CSV

In [ ]:
import pandas as pd

out_path = "experiments/taskB/results_summary.csv"
df.to_csv(out_path, index=False)
summary.to_csv(out_path.replace(".csv", "_per_model.csv"))
print(f"[Saved] {out_path}")
print(df.to_markdown())
print("\n[Next] Run evaluate.py for full test-set metrics + confusion matrix")